# GOLD (Dados Agregados)
Métricas e agregações<br>
Dados prontos para análise<br>
Otimizados para consulta<br>

## PROCESSAMENTO DOS DADOS

### IMPORTAÇÃO DAS BIBLIOTECAS

In [10]:
from pathlib import Path
from pyspark.sql import functions as F
from spark_utils import get_spark, write_single_csv

spark = get_spark("GoldLayer")


### CARREGAMENTO DOS DADOS

In [11]:
silver_file = Path("data/silver/dados_limpos.csv")
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(silver_file))
    .cache()
)
print(f"Silver carregado ({df.count()} registros).")


Silver carregado (10476 registros).


25/11/10 20:28:57 WARN CacheManager: Asked to cache already cached data.


## CRIAÇÃO DE NOVAS FEATURES

In [12]:
idade_expr = (
    F.when(F.col("IDADE") <= 20, "Até 20")
     .when((F.col("IDADE") > 20) & (F.col("IDADE") <= 30), "21 a 30")
     .when((F.col("IDADE") > 30) & (F.col("IDADE") <= 45), "31 a 45")
     .when((F.col("IDADE") > 45) & (F.col("IDADE") <= 55), "46 a 55")
     .otherwise("Maior que 55")
)
df = df.withColumn("FAIXA_ETARIA", idade_expr)
df.groupBy("FAIXA_ETARIA").count().show()


+------------+-----+
|FAIXA_ETARIA|count|
+------------+-----+
|      Até 20|  558|
|     21 a 30| 2994|
|Maior que 55| 1692|
|     46 a 55| 2664|
|     31 a 45| 2568|
+------------+-----+



In [13]:
df = df.withColumn(
    "TEM_FILHOS",
    F.when(F.col("QT_FILHOS") > 0, F.lit("Sim")).otherwise(F.lit("Não"))
)
df.groupBy("TEM_FILHOS").count().show()


+----------+-----+
|TEM_FILHOS|count|
+----------+-----+
|       Não| 3329|
|       Sim| 7147|
+----------+-----+



In [14]:
df = df.withColumn(
    "RENDA_TOTAL",
    F.coalesce(F.col("ULTIMO_SALARIO"), F.lit(0.0)) + F.coalesce(F.col("OUTRA_RENDA_VALOR"), F.lit(0.0))
)
df.select("RENDA_TOTAL").summary().show()


+-------+------------------+
|summary|       RENDA_TOTAL|
+-------+------------------+
|  count|             10476|
|   mean| 8927.768232149676|
| stddev|6242.9718171259465|
|    min|            1800.0|
|    25%|            3900.0|
|    50%|            8500.0|
|    75%|           11500.0|
|    max|           24400.0|
+-------+------------------+



In [15]:
df.groupBy(F.round(F.col("RENDA_TOTAL"), -2).alias("RENDA_TOTAL_FAIXA")).count().orderBy(F.col("RENDA_TOTAL_FAIXA")).show(10)


+-----------------+-----+
|RENDA_TOTAL_FAIXA|count|
+-----------------+-----+
|           1800.0|  846|
|           2200.0|  792|
|           3100.0|  792|
|           3900.0|  792|
|           4500.0|  468|
|           4800.0|  792|
|           6100.0|  524|
|           8500.0|  522|
|           9000.0|  522|
|           9100.0|    1|
+-----------------+-----+
only showing top 10 rows


In [16]:
df = df.withColumn(
    "CATEGORIA_RENDA",
    F.when(F.col("RENDA_TOTAL") <= 2500, "Baixa")
     .when(F.col("RENDA_TOTAL") <= 5000, "Média-Baixa")
     .when(F.col("RENDA_TOTAL") <= 10000, "Média")
     .when(F.col("RENDA_TOTAL") <= 20000, "Média-Alta")
     .otherwise("Alta")
)
df.groupBy("CATEGORIA_RENDA").count().show()


+---------------+-----+
|CATEGORIA_RENDA|count|
+---------------+-----+
|           Alta|  468|
|    Média-Baixa| 2844|
|          Média| 2648|
|          Baixa| 1638|
|     Média-Alta| 2878|
+---------------+-----+



In [17]:
bool_cols = ["TEM_FILHOS", "TRABALHANDO_ATUALMENTE", "CASA_PROPRIA"]
for col in bool_cols:
    normalized = F.upper(F.trim(F.col(col).cast("string")))
    df = df.withColumn(
        col,
        F.when(normalized.isin("SIM", "TRUE", "1"), F.lit(1)).otherwise(F.lit(0))
    )
print("Colunas booleanas convertidas para indicadores numéricos.")


Colunas booleanas convertidas para indicadores numéricos.


In [18]:
gold_path = "data/gold/dados_gold.csv"
write_single_csv(df, gold_path)
print(f"Dados gerais salvos: {gold_path} (shape=({df.count()}, {len(df.columns)}))")


Dados gerais salvos: data/gold/dados_gold.csv (shape=(10476, 24))


# AGREGAÇÃO Análise de Métricas

In [19]:
metricas_estado = (
    df.groupBy("UF")
      .agg(
          F.count("CODIGO_CLIENTE").alias("total_clientes"),
          F.avg("RENDA_TOTAL").alias("renda_media"),
          F.avg("SCORE").alias("score_medio"),
          F.avg("QT_IMOVEIS").alias("media_imoveis"),
          F.avg("QT_CARROS").alias("media_carros"),
          F.avg("TEM_FILHOS").alias("percentual_com_filhos")
      )
      .withColumn("percentual_com_filhos", F.col("percentual_com_filhos") * 100)
      .orderBy("UF")
)
write_single_csv(metricas_estado, "data/gold/metricas_estado.csv")
metricas_estado.show(5, truncate=False)


+---+--------------+-----------------+------------------+------------------+------------------+---------------------+
|UF |total_clientes|renda_media      |score_medio       |media_imoveis     |media_carros      |percentual_com_filhos|
+---+--------------+-----------------+------------------+------------------+------------------+---------------------+
|MG |1620          |8610.0           |48.955555555555556|0.7               |0.9555555555555556|78.88888888888889    |
|PR |1890          |8905.714285714286|50.59047619047619 |0.9142857142857143|1.0               |60.0                 |
|RJ |2916          |9789.883401920439|52.82098765432099 |0.9074074074074074|0.9444444444444444|70.37037037037037    |
|SC |1620          |8897.777777777777|47.7              |0.9               |0.9555555555555556|68.88888888888889    |
|SP |2430          |8142.222222222223|51.592592592592595|0.7851851851851852|0.8518518518518519|64.48559670781893    |
+---+--------------+-----------------+------------------

In [20]:
analise_clientes = (
    df.select(
        "CODIGO_CLIENTE", "IDADE", "FAIXA_ETARIA", "RENDA_TOTAL",
        "CATEGORIA_RENDA", "SCORE", "QT_IMOVEIS", "QT_CARROS",
        "ULTIMO_SALARIO", "TRABALHANDO_ATUALMENTE"
    )
    .withColumn(
        "capacidade_credito",
        F.coalesce(F.col("RENDA_TOTAL"), F.lit(0.0)) * F.lit(0.3) + F.coalesce(F.col("SCORE"), F.lit(0.0)) * F.lit(10)
    )
)
write_single_csv(analise_clientes, "data/gold/analise_clientes.csv")


In [21]:
ativos = (
    df.groupBy("FAIXA_ETARIA")
      .agg(
          F.avg("QT_IMOVEIS").alias("media_qt_imoveis"),
          F.avg("VL_IMOVEIS").alias("media_valor_imoveis"),
          F.avg("QT_CARROS").alias("media_qt_carros"),
          F.avg("VALOR_TABELA_CARROS").alias("media_valor_carros"),
          F.avg("RENDA_TOTAL").alias("renda_media"),
          F.avg("SCORE").alias("score_medio")
      )
      .orderBy("FAIXA_ETARIA")
)
write_single_csv(ativos, "data/gold/ativos_patrimonio.csv")


In [22]:
df.show(5, truncate=False)


+--------------+---+-----+---------------------+------------+---------+------------+----------+----------+-----------+-----------------+--------------------------+----------------------+--------------+---------+-------------------+-----+-----------------------+------------------+-----------+-----------------------+------------+----------+---------------+
|CODIGO_CLIENTE|UF |IDADE|ESCOLARIDADE         |ESTADO_CIVIL|QT_FILHOS|CASA_PROPRIA|QT_IMOVEIS|VL_IMOVEIS|OUTRA_RENDA|OUTRA_RENDA_VALOR|TEMPO_ULTIMO_EMPREGO_MESES|TRABALHANDO_ATUALMENTE|ULTIMO_SALARIO|QT_CARROS|VALOR_TABELA_CARROS|SCORE|DATA_UPLOAD            |ARQUIVO_FONTE     |RENDA_TOTAL|DATA_TRATAMENTO        |FAIXA_ETARIA|TEM_FILHOS|CATEGORIA_RENDA|
+--------------+---+-----+---------------------+------------+---------+------------+----------+----------+-----------+-----------------+--------------------------+----------------------+--------------+---------+-------------------+-----+-----------------------+------------------+------

In [23]:
print("Agregações criadas e salvas na camada Gold")


Agregações criadas e salvas na camada Gold
